# PAI最終課題：ACT の学習（Colab / L4 GPU）

Docker コンテナで変換した LeRobotDataset を Google Drive 経由で受け取り、ACT を学習します。

- **最終版**（関節角度＋天井カメラ、200本）：`crane_reach_image_v2` → `ckpt/act_image_v2`
- 比較用：画像あり（60本）`crane_reach_image`、画像なし（60本）`crane_reach_state`

事前準備（Drive の `MyDrive/pai_final/` に次を置く。場所を変えた場合は下の `DRIVE_DIR` を書き換える）:
```
pai_final/
  pai-final-crane-reach/          ← このリポジトリ（training/ フォルダを使う）
  datasets/crane_reach_image_v2/  ← convert.py --mode image の出力（200本）
  datasets/crane_reach_image/     ← 比較用（60本、画像あり）
  datasets/crane_reach_state/     ← 比較用（60本、画像なし）
```
メニューの「ランタイム → ランタイムのタイプを変更」で **L4 GPU** を選んでから実行してください。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/pai_final'
REPO_DIR = f'{DRIVE_DIR}/pai-final-crane-reach'
CKPT_DIR = f'{DRIVE_DIR}/ckpt'
RESULT_DIR = f'{DRIVE_DIR}/results'

## 1. LeRobot のインストール
教材（matsuolab/pai_training）が使っているのと**同じコミット**を入れます。数分かかります。
「ランタイムを再起動」を求められたら再起動し、**ドライブのマウントのセルだけ再実行**してから、このセルは飛ばして先へ進んでください。

In [ ]:
# datasets と draccus は教材の環境（pixi.lock）と同じ版に固定する。
# 新しい版だとデータの読み込みやモデル設定の保存・読み込みが失敗する。
!pip install -q "lerobot @ git+https://github.com/huggingface/lerobot@768e36660d1408c71118a2760f831c037fbfa17d" \
    "datasets==3.6.0" "draccus==0.10.0"
import lerobot, torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())

## 2. データセットをローカルディスクへコピー
Drive 上のまま読むと遅いので、Colab のディスクにコピーしてから学習します。

In [ ]:
!mkdir -p /content/data
!cp -r {DRIVE_DIR}/datasets/crane_reach_image_v2 /content/data/
# 比較用モデルも学習する場合
!cp -r {DRIVE_DIR}/datasets/crane_reach_image /content/data/
!cp -r {DRIVE_DIR}/datasets/crane_reach_state /content/data/
!cat /content/data/crane_reach_image_v2/meta/info.json | head -40

## 3. 学習（最終版：画像あり・200本）
約50分かかります。チェックポイントは Drive に保存されるので、途中で切断されても残ります。

In [ ]:
!python {REPO_DIR}/training/train.py \
    --dataset-root /content/data/crane_reach_image_v2 --repo-id crane_reach_image_v2 \
    --out {CKPT_DIR}/act_image_v2 --steps 30000 --batch-size 32 --val-episodes 10

## 3b. 比較用：画像あり（60本）
`--steps` は最初 2000 程度で動作確認し、問題なければ 20000 に増やしてください。

In [ ]:
STEPS = 20000
!python {REPO_DIR}/training/train.py \
    --dataset-root /content/data/crane_reach_image --repo-id crane_reach_image \
    --out {CKPT_DIR}/act_image --steps {STEPS} --batch-size 32

## 4. 比較用：画像なし（60本・ベースライン）


In [ ]:
!python {REPO_DIR}/training/train.py \
    --dataset-root /content/data/crane_reach_state --repo-id crane_reach_state \
    --out {CKPT_DIR}/act_state --steps 20000 --batch-size 32

## 5. 学習曲線

In [ ]:
import json
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
import os
for name in ['act_image_v2', 'act_image', 'act_state']:
    if not os.path.exists(f'{CKPT_DIR}/{name}/history.json'):
        continue
    h = json.load(open(f'{CKPT_DIR}/{name}/history.json'))
    ax.plot([r['step'] for r in h], [r['train_loss'] for r in h], label=f'{name} train', alpha=0.6)
    v = [r for r in h if 'val_loss' in r]
    ax.plot([r['step'] for r in v], [r['val_loss'] for r in v], 'o-', label=f'{name} val')
ax.set_yscale('log'); ax.set_xlabel('step'); ax.set_ylabel('loss'); ax.legend()
import os
os.makedirs(RESULT_DIR, exist_ok=True)
fig.savefig(f'{RESULT_DIR}/loss_curve.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. オフライン評価（教材の debug-policy の拡張版）
データセットの観測を方策に与え、予測した行動と正解（お手本）の行動を比べます。
検証用エピソード（学習に使っていない）での誤差が、汎化の目安になります。

In [ ]:
import os
for name, ds in [('act_image_v2', 'crane_reach_image_v2'), ('act_image', 'crane_reach_image'), ('act_state', 'crane_reach_state')]:
    if not os.path.exists(f'{CKPT_DIR}/{name}/last'):
        continue
    !python {REPO_DIR}/training/debug_policy.py \
        --dataset-root /content/data/{ds} --repo-id {ds} \
        --policy-path {CKPT_DIR}/{name}/last --out-dir {RESULT_DIR}/{name}

In [ ]:
import os
import glob
from IPython.display import Image, display
for name in ['act_image_v2', 'act_image', 'act_state']:
    if not os.path.exists(f'{RESULT_DIR}/{name}/offline_eval.json'):
        continue
    s = json.load(open(f'{RESULT_DIR}/{name}/offline_eval.json'))['summary']
    print(name, json.dumps(s, indent=1, ensure_ascii=False))
    for p in sorted(glob.glob(f'{RESULT_DIR}/{name}/episode_*.png'))[:2]:
        display(Image(p, width=500))

## 7. 次の手順
`ckpt/act_image_v2/last` をダウンロードし、Docker コンテナの `~/checkpoints/act_image_v2/` にコピーして、
`infer_node.py --temporal-ensemble 0.01`（推論）と `ros2 run pai_reach_sim evaluate`（成功率の測定）を実行します。
手順はリポジトリの README を参照してください。